You're given a dataset of searches for properties on Airbnb. For simplicity, let's say that each search result (i.e., each row) represents a unique host. Find the city with the most amenities across all their host's properties. Output the name of the city.

In [0]:
%sql
WITH amenities_count AS (
    SELECT 
        city,
        LENGTH(amenities) - LENGTH(REPLACE(amenities, ',', '')) + 1 AS amenity_count
    FROM airbnb_searches
),
city_totals AS (
    SELECT 
        city,
        SUM(amenity_count) AS total_amenities
    FROM amenities_count
    GROUP BY city
),
max_val AS (
    SELECT MAX(total_amenities) AS max_amenities
    FROM city_totals
)
SELECT city
FROM city_totals
WHERE total_amenities = (SELECT max_amenities FROM max_val);

In [0]:
from pyspark.sql import functions as F

# Count amenities per row
df = df.withColumn(
    "amenity_count",
    F.length("amenities") - F.length(F.regexp_replace("amenities", ",", "")) + 1
)

# Aggregate per city
city_totals = df.groupBy("city").agg(
    F.sum("amenity_count").alias("total_amenities")
)

# Get max value
max_val = city_totals.agg(F.max("total_amenities")).collect()[0][0]

# Filter cities with max
result = city_totals.filter(F.col("total_amenities") == max_val)

result.show()